# POC: DiT

📖 対応記事: `親子対話：DiT`🔗 [記事を読む](https://github.com/bonsai/sound-gen/blob/main/articles/%E8%A6%AA%E5%AD%90%E5%AF%BE%E8%A9%B1%EF%BC%9ADiT.md)

In [ ]:
# @title Setup
!pip install -q torch numpy matplotlib

In [ ]:
# @title Demo
# Diffusion Transformer (DiT) for Audio
import torch
import torch.nn as nn

# Minimal DiT block
class DiTBlock(nn.Module):
  def __init__(self, dim=64, nhead=4):
    super().__init__()
    self.norm1 = nn.LayerNorm(dim)
    self.attn = nn.MultiheadAttention(dim, nhead, batch_first=True)
    self.norm2 = nn.LayerNorm(dim)
    self.mlp = nn.Sequential(
      nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim)
    )
    # Adaptive LayerNorm (for timestep conditioning)
    self.adaLN = nn.Sequential(
      nn.SiLU(), nn.Linear(dim, 6*dim)
    )
  
  def forward(self, x, t_emb):
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp =       self.adaLN(t_emb).chunk(6, dim=-1)
    
    # Attention with conditioning
    x_norm = self.norm1(x) * (1 + scale_msa).unsqueeze(1) + shift_msa.unsqueeze(1)
    x = x + gate_msa.unsqueeze(1) * self.attn(x_norm, x_norm, x_norm)[0]
    
    # MLP with conditioning
    x_norm = self.norm2(x) * (1 + scale_mlp).unsqueeze(1) + shift_mlp.unsqueeze(1)
    x = x + gate_mlp.unsqueeze(1) * self.mlp(x_norm)
    return x

# Build tiny DiT
dim = 64
block = DiTBlock(dim)
x = torch.randn(2, 16, dim)
t = torch.randn(2, dim)
out = block(x, t)
print(f"DiT Block: {x.shape} → {out.shape}")
print(f"✅ DiT: Transformer replaces U-Net in diffusion")
print(f"   Scaled to billion params, achieves SOTA (e.g. Stable Diffusion 3)")

---
### 📝 まとめ
TODO

🔗 [記事を読む](https://github.com/bonsai/sound-gen/blob/main/articles/%E8%A6%AA%E5%AD%90%E5%AF%BE%E8%A9%B1%EF%BC%9ADiT.md)